<a href="https://colab.research.google.com/github/FilipeSchweitzer/ProjetoAplicado2_CDIA_UniSenai/blob/main/IST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd

def tables_preparation():
    df_ident = pd.read_excel('IDENTIFICACAO.xlsx', usecols=['Compound', 'Compound ID', 'Formula', 'Fragmentation Score', 'Score', 'Isotope Similarity', 'Description'])
    df_abund = pd.read_excel('ABUND.xlsx', usecols=['Compound', 'Identifications'])

    df = pd.merge(df_ident, df_abund, on='Compound')

    csv = df.to_csv('table.csv', index=False)

    return csv

df = pd.read_csv('table.csv')

In [6]:
#slide prof
def Pubchem_prof(name:str):
    import requests

    url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/JSON'

    response = requests.get(url, timeout=10)
    code = response.status_code

    if code == 200:
        data = response.json()
        props = data['PC_Compounds'][0]['props']
        return {p['urn']['label']: p['value'] for p in props}

    else:
        match code:
            case 400:
                return f'Error: {code} BAD REQUEST, name might be wrong, please check again!'
            case _:
                return f'Error: {code}'

In [7]:
tabela_final = pd.DataFrame(columns=['Compound ID', 'Description', 'IUPAC Name', 'Formula', 'SMILES', 'Fragmentation Score', 'Score', 'Isotope Similarity',
                        'Identifications'])

def add_to_final(index, info):
    original_table = df.iloc[index].fillna('Nan')
    api_search = info

    # print(original_table)

    data = []
    for column in tabela_final.columns:
        if column in original_table.index:
            #print(f'{column}: {original_table.loc[column]}')
            data.append(original_table[column])
        else:
            if info is str:
                data.append(info)
            elif column in info.keys():
                #print(f'{column}: {info[column]['sval']}')
                data.append(info[column]['sval'])
            else:
                data.append('Nan')

    print(data)

    # adiciona os valores a tabela final
    if len(data) == len(tabela_final.columns):
        tabela_final.loc[len(tabela_final)] = data
    else:
        print(f'\nHÁ COLUNAS EXTRAS !!!\n {data}\n{tabela_final.columns}')

In [8]:
#exvazia a tabela
def clear_table():
    import os
    global tabela_final
    tabela_final = tabela_final.iloc[0:0]

    arquivo = 'final.csv'
    if os.path.exists(arquivo):
        os.remove(arquivo)

clear_table()

In [ ]:
clear_table()

from time import sleep
from numpy import array_split #divide o dataframe a cada 100 linhas

found = 0

for index in range(50, 60):
    try:
        compound = df.iloc[index]['Description']

        search = Pubchem_prof(compound)

        if search != 'Error: 404':
            found += 1
            print(f'{index}.{compound} / found:{found}')
            add_to_final(index, search)
    except Exception as e:
        print(f"\nERRO!!! índice:{index}, erro:{e}\n")
        continue
    sleep(0.2)

#tabela_final.to_excel('final.xlsx', index=False)
tabela_final.to_csv('final.csv', index=False)
print('finished')